# Exercise: Cross Validation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

import sys
import os
sys.path.append(os.path.abspath('..')) # add parent directory to search path for modules

from src.utilities import read_filter_data

# Read data, filter rows with missing target, separate target from predictors
X_full, X_test_full, y = read_filter_data()

# Select numeric columns only
numeric_cols = [
    cname for cname in X_full.columns
    if X_full[cname].dtype in ['int64', 'float64']
]

X = X_full[numeric_cols].copy() # copy of X_full with only numeric columns
X_test = X_test_full[numeric_cols].copy() # copy of X_test_full with only numeric columns

In [ ]:
X.head()

So far we've learned to build pipelines with scikit-learn. For instance, the pipeline below will use SimpleImputer() to replace missing values in the data, before using RandomForestRegressor() to train a random forest model to make predictions. We set the number of trees in the random forest modle with the 'n_estimators' parameters, and setting 'random_state' to ensure reproducibility.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

my_pipeline = Pipeline(
    steps=[
        ('preprocessor', SimpleImputer()),
        ('Model', RandomForestRegressor(n_estimators=50, random_state=0))
    ]
)

We've also learned to use pipelines in cross-validation. The code below uses the cross_val_score() functions to obtain the MAE, averaged across five different folds. Recall we set the numbers of fold with the 'cv' parameter.

In [ ]:
from sklearn.model_selection import cross_val_score

# Multiply by -1 since sklearn calculuates negative MAE
scores = -1 * cross_val_score(
    my_pipeline,
    X,
    y,
    cv=5,
    scoring='neg_mean_absolute_error'
)

print("Average MAE scores:", scores.mean())

This function get_score() reports the average (over 3 cross-validation folds) MAE of a machine learning pipeline that uses:
- the data in X and y to create folds
- SimpleImputer() (with all parameters left as default) to replace missing values
- RandomForestRegressor() (with random_state=0) to fit a random forest model

The n_estimators parameter supplied to get_score is used when setting the numbers of trees in the random forest model.

In [ ]:
def get_score(n_estimators):
    """
    Return the average MAE over 3 CV folds of random forest model.

    Keyword argument:
    n_estimators -- the number of trees in the forest
    """
    pipeline = Pipeline(
        steps=[
            ('preprocessor', SimpleImputer()),
            ('model', RandomForestRegressor(n_estimators=n_estimators, random_state=0, n_jobs=-1))
        ]
    )

    scores = -1 * cross_val_score(
        pipeline,
        X,
        y,
        cv=3,
        scoring='neg_mean_absolute_error'
    )

    return scores.mean()

get_score() will now be used to evaluate the mode performance corresponding to eight different values for the number of trees in the random forest: 50, 100, 150, 200, 250, 300, 350, 400.

Results will be stored in a dictionary 'results' where results[i] is the average MAE returned by get_score(i).

In [ ]:
results = {} # dictionary to store results in (n_estimators:score) pairs

for i in range(50, 401, 50): # loop from 50 to 400, incrementing by 50
    score = get_score(i) # get score for each value of n_estimators
    results[i] = score # add (n_estimators:score) pair to results

This cell visualizes the results above

In [ ]:
import matplotlib.pyplot as plt

# display plots directly inside the notebook
%matplotlib inline 

# plot results, n_estimators on x-axis, scores on y-axis
plt.plot(list(results.keys()), list(results.values()))
plt.show() # display plot

In [ ]:
# lowest MAE in results
n_estimators_best = min(results, key=results.get) # get key with lowest MAE
# key=results.get compares score instead of n_estimators in (n_esimators:score) pairs
best_score = results[n_estimators_best]

print(f"Best n_estimators: {n_estimators_best}, score: {best_score}")

This exercise explored one method for choosing appropriate parameters in a machine learning model.
Going further with hyperparameter optimization, 'grid search' is a straightforward strategy for determining the best combination of parameters of a machine learning model. Scikit-learn also contains a function GridSearchCV() that can make grid search code more efficient.